<a href="https://colab.research.google.com/github/gokila-a/Resume_Screening_Using_RAG_ChromaDB_LLM/blob/main/Resume_Screening_Using_RAG_ChromaDB_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q chromadb sentence-transformers pypdf langchain langchain-community transformers accelerate bitsandbytes langchain-text-splitters

In [ ]:
import chromadb
import sentence_transformers
import transformers

print("All libraries loaded successfully!")

All libraries loaded successfully!


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving JD RESUME.pdf to JD RESUME (1).pdf
Saving abi_final _resumee.pdf to abi_final _resumee (1).pdf
Saving PONMI RESUME .pdf to PONMI RESUME  (1).pdf


In [ ]:
from pypdf import PdfReader

documents = []

# Use the actual uploaded filenames
pdf_files = list(uploaded.keys())

for pdf in pdf_files:
    reader = PdfReader(pdf)

    text = ""

    for page in reader.pages:
        text += page.extract_text()

    documents.append(text)

print("Loaded", len(documents), "resumes")

Loaded 3 resumes


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = []

for doc in documents:
    chunks.extend(splitter.split_text(doc))

print("Total Chunks:", len(chunks))

Total Chunks: 15


In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

embeddings = embedding_model.encode(chunks)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
import chromadb

client = chromadb.Client()

collection = client.create_collection(
    name="resume_collection"
)

for i, chunk in enumerate(chunks):
    collection.add(
        ids=[str(i)],
        documents=[chunk],
        embeddings=[embeddings[i].tolist()]
    )

print("Stored in ChromaDB")

Stored in ChromaDB


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=200
)

print("LLM Loaded")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LLM Loaded


In [ ]:
def retrieve(query, top_k=3):

    query_embedding = embedding_model.encode([query])[0]

    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k
    )

    return results["documents"][0]

In [ ]:
def rag(query):

    retrieved_docs = retrieve(query)

    context = "\n\n".join(retrieved_docs)

    prompt = f"""
Use the resume information below.

Context:
{context}

Question:
{query}

Answer:
"""

    result = generator(prompt)

    return result[0]["generated_text"]

In [ ]:
query = "Who has experience in Machine Learning?"

answer = rag(query)

print(answer)

[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



Use the resume information below.

Context:
location-based services and APIs. 
• It integrates map services to display caf´es visually and offers features like search suggestions, distance filtering, and route 
navigation. 
CERTIFICATIONS 
• Data Analysis with Python – IBM (2025) 
• Machine Learning – Mathworks (2025) 
• Pointers in C – Infosys (2024) 
• Learn AI and Gen AI Basics – Microsoft(2024) 
TECHNICAL SKILLS 
Languages      Python, HTML, CSS, SQL, JavaScript,c++

GOVT ADW High School, Athikaram 2021 – 2022 
Secondary Education (Percentage: 72%) Sivagangai, Tamil Nadu 
CERTIFICATIONS 
Python for Data Science Jul 2025 – Aug 2025 
IIT Madras — NPTEL 
• Completed a 4-week course covering Python for data analysis, visualization, and scientific computing. 
Deep Learning with TensorFlow Aug 2025 
CognitiveClass.ai (IBM Developer Skills Network) 
• Completed an IBM-powered course on deep learning fundamentals using TensorFlow. 
Projects

• Developed a Voice-Based AI Assistant to autom

In [ ]:
rag("Who knows Python?")

[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'\nUse the resume information below.\n\nContext:\nlocation-based services and APIs. \n• It integrates map services to display caf´es visually and offers features like search suggestions, distance filtering, and route \nnavigation. \nCERTIFICATIONS \n• Data Analysis with Python – IBM (2025) \n• Machine Learning – Mathworks (2025) \n• Pointers in C – Infosys (2024) \n• Learn AI and Gen AI Basics – Microsoft(2024) \nTECHNICAL SKILLS \nLanguages      Python, HTML, CSS, SQL, JavaScript,c++\n\nGOVT ADW High School, Athikaram 2021 – 2022 \nSecondary Education (Percentage: 72%) Sivagangai, Tamil Nadu \nCERTIFICATIONS \nPython for Data Science Jul 2025 – Aug 2025 \nIIT Madras — NPTEL \n• Completed a 4-week course covering Python for data analysis, visualization, and scientific computing. \nDeep Learning with TensorFlow Aug 2025 \nCognitiveClass.ai (IBM Developer Skills Network) \n• Completed an IBM-powered course on deep learning fundamentals using TensorFlow. \nProjects\n\nusing Python and Mac

In [ ]:
rag("Compare all three candidates")

[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'\nUse the resume information below.\n\nContext:\nVetri vikaas public school, Rasipuram 2021 – 2022 \nSecondary Education (Percentage: 93%) Namakkal, Tamil Nadu \n \nCERTIFICATIONS \nMongoDB for Data Analysts Certification Jun 2025 \nMongoDB \n• Focused on data modeling, querying, and aggregation techniques for analytical tasks. \nProject on Microsoft  Word Jul 2025 \nMicrosoft \n• Certified in Microsoft Word with emphasis on data analysis documentation and report creation. \nPROJECT \nImage Classification  using Python Sep 2025\n\nSSLC-                                                                                                                                                                                      (2021 – 2022) \nSri Sankara Vidyalaya hr . sec. School, Puducherry                                                                                                        Percentage: 72 \nPROJECTS \nInteractive Photo Gallery — HTML, CSS, JS\n\nHSC – computer science          

In [ ]:
rag("Which resume is best suited for Data Scientist role?")

[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'\nUse the resume information below.\n\nContext:\nJAYADARSINI RJ \n(+91) 8072620963 \njayadarsinijayaraman@gmail.com \n https://www.linkedin.com/in/jayadarsini-rj-599258366  https://github.com/Jayadarsini-hub \n \nEDUCATION \nKarpagam College of Engineering, Coimbatore 2026 – Pursuing \nB.Tech in Artificial Intelligence and Data Science (CGPA: 8.3 upto 3rd semester) Coimbatore, Tamil Nadu \nVetri vikaas public school, Rasipuram 2023 – 2024 \nHigher Secondary Education (Percentage: 93%) Namakkal, Tamil Nadu\n\nlocation-based services and APIs. \n• It integrates map services to display caf´es visually and offers features like search suggestions, distance filtering, and route \nnavigation. \nCERTIFICATIONS \n• Data Analysis with Python – IBM (2025) \n• Machine Learning – Mathworks (2025) \n• Pointers in C – Infosys (2024) \n• Learn AI and Gen AI Basics – Microsoft(2024) \nTECHNICAL SKILLS \nLanguages      Python, HTML, CSS, SQL, JavaScript,c++\n\nGOVT ADW High School, Athikaram 2021 – 202